# Data Cleaning and Quality Checks 🧹

### ISA 383: Python for Business Analytics

Real data is rarely ready for analysis. This notebook shows a small, reproducible cleaning
workflow: identify the problem, make an explicit decision, apply the change, and validate the
result.


## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.


## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Detect duplicates, missing values, inconsistent strings, and incorrect data types.
2. Convert dates and numeric strings safely with `errors='coerce'`.
3. Choose and document a reasonable treatment for missing values.
4. Flag unusual observations without assuming that every outlier is an error.
5. Use `drop_duplicates()`, `fillna()`, `quantile()`, and `transform()` in a repeatable cleaning workflow.


## 1. Start with the source data 🌤️

The source file is the same frozen Dubai weather extract used in the pandas notebook. The
data came from a real public API. To teach data cleaning without modifying the source, we will
make a very small teaching copy and introduce common operational problems into that copy.

This distinction matters: a teaching error is not evidence that the original source was wrong.


### 🧭 Syntax first: Reading the source file

Read the pattern before running the example. The names in the pattern are placeholders.

```python
pd.read_csv(
    filepath,
    parse_dates=["date"],
    na_values=None,
)
```

Documentation: [read_csv documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from IPython.display import display

REPO_RAW = "https://raw.githubusercontent.com/harshvardhaniimi/isa383-google-colab/main"
DATA_DIR = Path("/content/isa383/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = DATA_DIR / "uae_weather_daily.csv"

urlretrieve(f"{REPO_RAW}/data/uae_weather_daily.csv", DATA_FILE)
weather_raw = pd.read_csv(DATA_FILE, parse_dates=["date"])
display(weather_raw.head())


The examples use a frozen, fourteen-day Dubai weather extract. The values were retrieved
from the Open-Meteo Historical Weather API for 1-14 January 2025 and stored locally so
that every student runs the same file. The file is small enough to inspect directly, but
still has dates, numeric measures, a key, units, and missing-data decisions to discuss.

Source: https://archive-api.open-meteo.com/v1/archive
Location: Dubai, UAE (approximately 25.20 N, 55.30 E)


> 💭 **Quick thought question:** Which problems would you fix before analysis, and which would you only document?


## 2. Create a compact messy copy 🧪

We will work with eight source rows. The copy contains a duplicate key, whitespace, numeric
values stored as text, one missing temperature, and one deliberately extreme wind value.
These are common patterns in files exported from different systems.


### 🧭 Syntax first: Making a small teaching copy

Read the pattern before running the example. The names in the pattern are placeholders.

```python
copy = df.copy()
copy.loc[row_selector, "column"] = new_value
copy["column"] = copy["column"].astype("string")
pd.concat([df1, df2], ignore_index=True)
```

Documentation: [copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)


In [2]:
messy_weather = weather_raw.head(8).copy()

# Store selected columns as text so that we can see the conversion problem.
messy_weather["temp_max_c"] = messy_weather["temp_max_c"].astype("object")
messy_weather["precipitation_mm"] = messy_weather["precipitation_mm"].astype("object")
messy_weather["wind_max_kmh"] = messy_weather["wind_max_kmh"].astype("object")

messy_weather.loc[1, "temp_max_c"] = "22.6 C"
messy_weather.loc[2, "temp_min_c"] = np.nan
messy_weather.loc[3, "city"] = " Dubai "
messy_weather.loc[4, "precipitation_mm"] = "0 mm"
messy_weather.loc[5, "wind_max_kmh"] = "99 km/h"

# A duplicate record can appear after two systems are combined.
messy_weather = pd.concat([messy_weather, messy_weather.iloc[[3]]], ignore_index=True)
messy_weather


Out[0]: 
        date station_id     city  ... temp_min_c  precipitation_mm wind_max_kmh
0 2025-01-01        DXB    Dubai  ...       18.4               0.7         21.7
1 2025-01-02        DXB    Dubai  ...       19.9               0.0         26.1
2 2025-01-03        DXB    Dubai  ...        NaN               0.0         17.3
3 2025-01-04        DXB   Dubai   ...       13.0               0.0         14.5
4 2025-01-05        DXB    Dubai  ...       15.6              0 mm         13.1
5 2025-01-06        DXB    Dubai  ...       12.5               0.0      99 km/h
6 2025-01-07        DXB    Dubai  ...       14.3               0.0         13.8
7 2025-01-08        DXB    Dubai  ...       14.0               0.0          9.9
8 2025-01-04        DXB   Dubai   ...       13.0               0.0         14.5

[9 rows x 7 columns]


## 3. Diagnose before changing anything 🔎

A quality check is not only a search for errors. It is a compact description of what the file
contains and where decisions are needed.


### 🧭 Syntax first: Checking missing values, duplicates, and cardinality

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.isna().sum()
df.nunique(dropna=False)
df.duplicated(subset=["key"], keep=False)
df.drop_duplicates(subset=["key"], keep="last")
```

Documentation: [isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) · [nunique](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html) · [duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)


In [3]:
def data_quality_report(df):
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "unique_values": df.nunique(dropna=False),
        }
    ).sort_values("missing", ascending=False)

data_quality_report(messy_weather)


Out[0]: 
                           dtype  missing  unique_values
temp_min_c               float64        1              8
station_id                object        0              1
date              datetime64[ns]        0              8
city                      object        0              2
temp_max_c                object        0              8
precipitation_mm          object        0              3
wind_max_kmh              object        0              8


In [4]:
duplicate_keys = messy_weather.duplicated(subset=["station_id", "date"], keep=False)
print("Duplicate key rows:", duplicate_keys.sum())
messy_weather.loc[duplicate_keys, ["station_id", "date", "city"]]


Duplicate key rows: 2
Out[0]: 
  station_id       date     city
3        DXB 2025-01-04   Dubai 
8        DXB 2025-01-04   Dubai 


> 💭 **Quick thought question:** Why is checking duplicates on the business key often more useful than checking whether every cell is identical?


## 4. Convert types and standardize text 🔤

`errors='coerce'` converts values that cannot be interpreted into `NaN`. That is safer than
silently keeping a column as text, but it creates a follow-up task: inspect the new missing values.


### 🧭 Syntax first: Converting types and cleaning text

Read the pattern before running the example. The names in the pattern are placeholders.

```python
pd.to_datetime(values, errors="coerce")
pd.to_numeric(values, errors="coerce")
series.astype("string").str.strip().str.title()
series.astype("string").str.extract(pattern)
df.convert_dtypes()
```

Documentation: [to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [str methods](https://pandas.pydata.org/docs/user_guide/text.html) · [str.extract](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html) · [convert_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html)


In [5]:
cleaned = messy_weather.copy()

cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
cleaned["city"] = cleaned["city"].astype("string").str.strip().str.title()

numeric_columns = ["temp_max_c", "temp_min_c", "precipitation_mm", "wind_max_kmh"]
for column in numeric_columns:
    number_text = cleaned[column].astype("string").str.extract(r"([-+]?\d*\.?\d+)")[0]
    cleaned[column] = pd.to_numeric(number_text, errors="coerce")

cleaned.dtypes


Out[0]: 
date                datetime64[ns]
station_id                  object
city                string[python]
temp_max_c                 Float64
temp_min_c                 Float64
precipitation_mm           Float64
wind_max_kmh               Float64
dtype: object


In [6]:
cleaned[["date", "city", "temp_max_c", "temp_min_c", "precipitation_mm", "wind_max_kmh"]].head()


Out[0]: 
        date   city  temp_max_c  temp_min_c  precipitation_mm  wind_max_kmh
0 2025-01-01  Dubai        25.0        18.4               0.7          21.7
1 2025-01-02  Dubai        22.6        19.9               0.0          26.1
2 2025-01-03  Dubai        21.5        <NA>               0.0          17.3
3 2025-01-04  Dubai        22.9        13.0               0.0          14.5
4 2025-01-05  Dubai        22.8        15.6               0.0          13.1


## 5. Missing values: a decision, not a reflex 🤔

Dropping a row, filling a value, and leaving a value missing are different decisions. The right
choice depends on what the variable means and how the result will be used.

Here, we use the station median for the missing minimum temperature only as a teaching example.
In a real analysis, we would document the reason and test whether the choice changes the result.


### 🧭 Syntax first: Dropping or filling missing values

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.dropna(subset=["column"])
df.fillna({"column": replacement_value})
df.groupby("group")["column"].transform("median")
```

Documentation: [dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) · [fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) · [transform](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.SeriesGroupBy.transform.html)


In [7]:
print("Missing values before imputation:")
print(cleaned[numeric_columns].isna().sum())

station_median = cleaned.groupby("station_id")["temp_min_c"].transform("median")
cleaned["temp_min_c"] = cleaned["temp_min_c"].fillna(station_median)

print("\nMissing values after the selected imputation:")
print(cleaned[numeric_columns].isna().sum())


Missing values before imputation:
temp_max_c          0
temp_min_c          1
precipitation_mm    0
wind_max_kmh        0
dtype: int64

Missing values after the selected imputation:
temp_max_c          0
temp_min_c          0
precipitation_mm    0
wind_max_kmh        0
dtype: int64


> 💭 **Quick thought question:** When would filling a missing precipitation value with zero be a dangerous assumption?


## 6. Outliers: flag first, investigate second ⚠️

The IQR rule is a useful screening tool. It does not tell us that an observation is wrong.
A real storm, sale, or operational failure can be unusual and still valid.


### 🧭 Syntax first: Measuring spread and flagging unusual values

Read the pattern before running the example. The names in the pattern are placeholders.

```python
series.quantile([0.25, 0.50, 0.75])
series.clip(lower=minimum, upper=maximum)
```

Documentation: [quantile](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html) · [clip](https://pandas.pydata.org/docs/reference/api/pandas.Series.clip.html)


In [8]:
cleaned["wind_max_kmh"].quantile([0.25, 0.50, 0.75]).round(1)


Out[0]: 
0.25    13.8
0.50    14.5
0.75    21.7
Name: wind_max_kmh, dtype: Float64


In [9]:
def iqr_outlier_flags(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

cleaned["wind_outlier_flag"] = iqr_outlier_flags(cleaned["wind_max_kmh"])
cleaned[["date", "wind_max_kmh", "wind_outlier_flag"]]


Out[0]: 
        date  wind_max_kmh  wind_outlier_flag
0 2025-01-01          21.7              False
1 2025-01-02          26.1              False
2 2025-01-03          17.3              False
3 2025-01-04          14.5              False
4 2025-01-05          13.1              False
5 2025-01-06          99.0               True
6 2025-01-07          13.8              False
7 2025-01-08           9.9              False
8 2025-01-04          14.5              False


## 7. Build a repeatable cleaning function 🧰

A function makes the cleaning logic easier to rerun on another file. It also makes the
assumptions visible to a reviewer.


### 🧭 Syntax first: Removing duplicates and restoring a clean index

Read the pattern before running the example. The names in the pattern are placeholders.

```python
df.drop_duplicates(subset=["key"], keep="last")
df.sort_values(["group", "date"])
df.reset_index(drop=True)
```

Documentation: [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) · [reset_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)


In [10]:
def clean_weather_data(df):
    out = df.copy()

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["city"] = out["city"].astype("string").str.strip().str.title()

    for column in ["temp_max_c", "temp_min_c", "precipitation_mm", "wind_max_kmh"]:
        number_text = out[column].astype("string").str.extract(r"([-+]?\d*\.?\d+)")[0]
        out[column] = pd.to_numeric(number_text, errors="coerce")

    out = out.drop_duplicates(subset=["station_id", "date"], keep="last")

    station_median = out.groupby("station_id")["temp_min_c"].transform("median")
    out["temp_min_c"] = out["temp_min_c"].fillna(station_median)
    out["temp_range_c"] = (out["temp_max_c"] - out["temp_min_c"]).round(1)
    out["wind_outlier_flag"] = iqr_outlier_flags(out["wind_max_kmh"])

    return out.sort_values(["station_id", "date"]).reset_index(drop=True)

clean_weather = clean_weather_data(messy_weather)
clean_weather


Out[0]: 
        date station_id   city  ...  wind_max_kmh  temp_range_c  wind_outlier_flag
0 2025-01-01        DXB  Dubai  ...          21.7           6.6              False
1 2025-01-02        DXB  Dubai  ...          26.1           2.7              False
2 2025-01-03        DXB  Dubai  ...          17.3           7.2              False
3 2025-01-04        DXB  Dubai  ...          14.5           9.9              False
4 2025-01-05        DXB  Dubai  ...          13.1           7.2              False
5 2025-01-06        DXB  Dubai  ...          99.0          12.6               True
6 2025-01-07        DXB  Dubai  ...          13.8          11.5              False
7 2025-01-08        DXB  Dubai  ...           9.9          10.1              False

[8 rows x 9 columns]


In [11]:
clean_weather.convert_dtypes().dtypes


Out[0]: 
date                 datetime64[ns]
station_id           string[python]
city                 string[python]
temp_max_c                  Float64
temp_min_c                  Float64
precipitation_mm            Float64
wind_max_kmh                Float64
temp_range_c                Float64
wind_outlier_flag           boolean
dtype: object


In [12]:
assert clean_weather["date"].notna().all()
assert clean_weather.duplicated(["station_id", "date"]).sum() == 0
assert clean_weather["temp_max_c"].dtype.kind in "fi"

print("Validation checks passed.")


Validation checks passed.


## Comprehensive exercises 📝

Use the small `messy_weather` table unless an exercise asks you to use `weather_raw`.
For every cleaning decision, write a brief reason. A technically valid transformation can
still be a poor analytical decision if its assumption is not explained.


### Exercise 1: Data-quality report

Write a `data_quality_report(df)` function that also reports the percentage of missing values and the number of duplicate station-date keys.


In [ ]:
# Write your solution here.


### Exercise 2: Robust string standardization

Create a small `condition` column with variants such as `rain`, `Rain `, and `CLEAR`. Standardize it to a consistent set of labels and explain your mapping.


In [ ]:
# Write your solution here.


### Exercise 3: A missing-value decision

Compare two treatments for the missing minimum temperature: dropping the row and median imputation. Show how the average changes and state which treatment you would defend for this small example.


In [ ]:
# Write your solution here.


### Exercise 4: Outlier detection

Write and test your own `detect_outliers_iqr(series)` function. Apply it to wind speed and explain why a flag is not the same as deleting the observation.


In [ ]:
# Write your solution here.


### Exercise 5: Full pipeline on a new small table

Create a six-row table of business transactions with one duplicate, one missing value, and one numeric value stored as text. Build a cleaning function, validate its output, and list the assumptions you made.


In [ ]:
# Write your solution here.


## Summary ✅

A defensible cleaning workflow is:

1. Describe the source before changing it.
2. Identify duplicates, missing values, types, and inconsistent labels.
3. Apply explicit, documented decisions.
4. Flag unusual values before deciding whether they are errors.
5. Validate the cleaned result and keep the transformation reproducible.

Cleaning improves the data structure. It does not automatically make the resulting analysis
correct or causal.
